# Validation experiments — trained Microbiome PFN

A battery of validation experiments run against a trained checkpoint:

1. **Held-out cell prediction** vs the marginal-mean baseline (Spearman ρ, NLL, 80% coverage).
2. **Predictive calibration** — empirical coverage of central intervals at several nominal levels.
3. **Generalization across dataset size** — metrics as the number of taxa T varies.
4. **Permutation-equivariance** of the *trained* weights (the load-bearing architectural claim).
5. **Compositional constraint** — predicted relative abundances sum to 1 per sample.
6. **Treatment-effect recovery** — counterfactual ranking vs known ground-truth responsive taxa.

> Set `CHECKPOINT` in the setup cell to validate your own model (e.g. one trained with `notebooks/train_t4.ipynb`). If left `None`, the notebook reuses a checkpoint in `checkpoints/` if present, otherwise trains a short treatment-aware demo so it runs standalone. **Demo-length training gives weak numbers** — a properly trained checkpoint (~10⁵–10⁶ steps) is what you actually want to validate. Use a GPU (Runtime → T4).

## 0. Setup

In [ ]:
%pip install -q "git+https://github.com/metagenAu/microbiomepfn.git"
import microbiomepfn
print("microbiomepfn", microbiomepfn.__version__)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU — this will be slow. Set Runtime > Change runtime type > T4 GPU.")

In [ ]:
import glob
import os
from microbiomepfn.train import train
from microbiomepfn.deploy import load_model

CHECKPOINT = None  # <-- set to a path to validate your own model

if CHECKPOINT is None:
    existing = sorted(glob.glob("checkpoints/model_step*.pt"), key=os.path.getmtime)
    if existing:
        CHECKPOINT = existing[-1]
        print("Reusing existing checkpoint:", CHECKPOINT)
    else:
        print("No checkpoint found — training a short treatment-aware demo model...")
        train(n_steps=3000, d=256, n_layers=6, n_heads=8,
              n_taxa_cap=400, n_samples_cap=140,
              device_str=device, save_dir="checkpoints",
              use_treatment=True, p_treatment_study=0.5,
              log_every=250, save_every=3000)
        CHECKPOINT = max(glob.glob("checkpoints/model_step*.pt"), key=os.path.getmtime)

model, cfg = load_model(CHECKPOINT, device=torch.device(device))
model.eval()
print("loaded:", CHECKPOINT)
print("config:", {k: cfg[k] for k in ["d", "n_layers", "n_heads", "k_phylo", "d_samp_in"]})

## 1. Held-out cell prediction vs baseline

Mask 15% of cells and predict them. Compare the model's Spearman ρ (predicted vs true count, log scale) and NLL against a marginal-mean baseline (per-taxon mean relative abundance × library size). **The model should beat the baseline ρ.**

In [ ]:
from microbiomepfn.eval import quick_eval

res = quick_eval(CHECKPOINT, n_eval=20, device=device, seed=2024)
nll = np.array([r["nll_per_cell"] for r in res])
rho_m = np.array([r["spearman_model"] for r in res])
rho_b = np.array([r["spearman_baseline"] for r in res])
cov80 = np.array([r["coverage_80"] for r in res])

print(f"draws            : {len(res)}")
print(f"NLL/cell         : {nll.mean():.3f} +/- {nll.std():.3f}")
print(f"model    rho     : {rho_m.mean():.3f} +/- {rho_m.std():.3f}")
print(f"baseline rho     : {rho_b.mean():.3f} +/- {rho_b.std():.3f}")
print(f"model beats base : {(rho_m > rho_b).mean()*100:.0f}% of draws")
print(f"80% coverage     : {cov80.mean():.3f}  (target 0.80)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
lim = [min(rho_b.min(), rho_m.min()) - 0.05, max(rho_b.max(), rho_m.max()) + 0.05]
ax[0].plot(lim, lim, "k--", lw=1)
ax[0].scatter(rho_b, rho_m)
ax[0].set_xlabel("baseline rho"); ax[0].set_ylabel("model rho")
ax[0].set_title("Model vs baseline (above line = model wins)")
ax[1].hist(cov80, bins=10)
ax[1].axvline(0.80, color="r", ls="--")
ax[1].set_xlabel("empirical 80% coverage"); ax[1].set_title("Interval calibration")
plt.tight_layout(); plt.show()

## 2. Predictive calibration curve

Empirical coverage of the model's central predictive intervals at several nominal levels (negative-binomial Monte-Carlo). A well-calibrated model sits on the diagonal.

In [ ]:
from microbiomepfn.prior import sample_dataset, PriorConfig
from microbiomepfn.features import build_batch
from microbiomepfn.train import batch_to_torch, _pad_sample_feats, _subsample_taxa


@torch.no_grad()
def coverage_at_levels(ds, levels, mask_frac=0.15, seed=0, n_mc=200):
    rng = np.random.default_rng(seed)
    batch = build_batch(ds, rng, k_phylo=cfg["k_phylo"],
                        cell_mask_frac=mask_frac, sample_query_frac=0.0)
    batch = _pad_sample_feats(batch, cfg["d_samp_in"])
    bt = batch_to_torch(batch, torch.device(device))
    log_lib = torch.log(bt["library_sizes"].float().clamp(min=1))
    out = model(cell_feats=bt["cell_feats"], tax_feats=bt["tax_feats"],
                samp_feats=bt["samp_feats"], visible_cell=bt["visible_cell"],
                log_library=log_lib)
    masked = (~bt["visible_cell"]).float()
    mu = out["log_mu"].exp()
    theta = out["log_theta"].exp()
    gamma = torch.distributions.Gamma(theta, theta / mu.clamp(min=1e-8))
    samples = torch.distributions.Poisson(gamma.sample((n_mc,))).sample()
    counts = bt["counts"]
    covs = []
    for lv in levels:
        lo = torch.quantile(samples, (1 - lv) / 2, dim=0)
        hi = torch.quantile(samples, 1 - (1 - lv) / 2, dim=0)
        inside = ((counts >= lo) & (counts <= hi)).float()
        covs.append(float((inside * masked).sum() / masked.sum().clamp(min=1)))
    return np.array(covs)


levels = np.array([0.5, 0.7, 0.8, 0.9, 0.95])
emp = []
for s in range(6):
    ds = sample_dataset(cfg=PriorConfig(), rng=np.random.default_rng(100 + s))
    if ds.counts.shape[1] > 300:
        idx = np.random.default_rng(s).choice(ds.counts.shape[1], 300, replace=False)
        ds = _subsample_taxa(ds, idx)
    emp.append(coverage_at_levels(ds, levels, seed=s))
emp = np.stack(emp).mean(0)
for lv, e in zip(levels, emp):
    print(f"nominal {lv:.2f} -> empirical {e:.3f}")

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.plot(levels, emp, "o-")
plt.xlabel("nominal coverage"); plt.ylabel("empirical coverage")
plt.title("Calibration"); plt.xlim(0, 1); plt.ylim(0, 1); plt.show()

## 3. Generalization across dataset size

Sweep the number of taxa T (samples held ~fixed) and re-measure held-out prediction. The architecture has no taxon-count-specific parameters, so quality should degrade gracefully, not cliff.

In [ ]:
from microbiomepfn.eval import evaluate_held_out_cells

Ts = [50, 100, 200, 400]
rows = []
for T in Ts:
    pc = PriorConfig()
    pc.n_taxa_range = (T, T + 1)
    pc.n_samples_range = (120, 121)
    rng = np.random.default_rng(7)
    mvals, bvals, nvals = [], [], []
    for _ in range(4):
        ds = sample_dataset(cfg=pc, rng=rng)
        m = evaluate_held_out_cells(model, ds, rng, d_samp_max=cfg["d_samp_in"],
                                    k_phylo=cfg["k_phylo"], device=torch.device(device))
        mvals.append(m["spearman_model"])
        bvals.append(m["spearman_baseline"])
        nvals.append(m["nll_per_cell"])
    rows.append((T, np.mean(mvals), np.mean(bvals), np.mean(nvals)))
    print(f"T={T:4d} | model rho {np.mean(mvals):.3f} | base rho {np.mean(bvals):.3f} | nll {np.mean(nvals):.3f}")

rows = np.array(rows)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rows[:, 0], rows[:, 1], "o-", label="model")
ax[0].plot(rows[:, 0], rows[:, 2], "s--", label="baseline")
ax[0].set_xlabel("T (taxa)"); ax[0].set_ylabel("Spearman rho"); ax[0].legend(); ax[0].set_title("rho vs size")
ax[1].plot(rows[:, 0], rows[:, 3], "o-")
ax[1].set_xlabel("T (taxa)"); ax[1].set_ylabel("NLL/cell"); ax[1].set_title("NLL vs size")
plt.tight_layout(); plt.show()

## 4. Permutation-equivariance (trained weights)

Permute the taxon axis of the inputs and confirm the trained model's per-taxon output `log_mu` permutes identically while the per-sample readout `y_pred` is invariant. Differences should be floating-point noise. **This is an architectural invariant — it must pass regardless of training quality.**

In [ ]:
ds = sample_dataset(cfg=PriorConfig(), rng=np.random.default_rng(0))
if ds.counts.shape[1] > 256:
    idx = np.random.default_rng(0).choice(ds.counts.shape[1], 256, replace=False)
    ds = _subsample_taxa(ds, idx)
rng = np.random.default_rng(0)
batch = build_batch(ds, rng, k_phylo=cfg["k_phylo"], cell_mask_frac=0.15, sample_query_frac=0.3)
batch = _pad_sample_feats(batch, cfg["d_samp_in"])
bt = batch_to_torch(batch, torch.device(device))
log_lib = torch.log(bt["library_sizes"].float().clamp(min=1))
T = bt["cell_feats"].shape[1]
perm = torch.randperm(T, device=torch.device(device))

with torch.no_grad():
    o1 = model(cell_feats=bt["cell_feats"], tax_feats=bt["tax_feats"], samp_feats=bt["samp_feats"],
               visible_cell=bt["visible_cell"], log_library=log_lib)
    o2 = model(cell_feats=bt["cell_feats"][:, perm, :], tax_feats=bt["tax_feats"][perm, :],
               samp_feats=bt["samp_feats"], visible_cell=bt["visible_cell"][:, perm], log_library=log_lib)

dmu = (o1["log_mu"][:, perm] - o2["log_mu"]).abs().max().item()
dy = (o1["y_pred"] - o2["y_pred"]).abs().max().item()
print(f"max |delta log_mu| (should permute)    : {dmu:.2e}")
print(f"max |delta y_pred| (should be invariant): {dy:.2e}")
assert dmu < 2e-3 and dy < 2e-3, "equivariance violated on trained weights!"
print("PASS — trained model is permutation-equivariant over taxa.")

## 5. Compositional constraint

The count head's per-sample relative abundances are a softmax over taxa, so they must sum to exactly 1 per sample.

In [ ]:
with torch.no_grad():
    rel = torch.softmax(o1["log_mu"], dim=-1)  # (N, T)
sums = rel.sum(dim=-1)
print(f"per-sample sum of relative abundance: min {sums.min().item():.6f}, max {sums.max().item():.6f}")
assert torch.allclose(sums, torch.ones_like(sums), atol=1e-4)
print("PASS — relative abundances sum to 1 per sample.")

## 6. Treatment-effect recovery vs ground truth

Force a treatment draw (ground-truth responsive taxa and effect signs are known), run the counterfactual treatment-effect ranking, and measure how well it matches the truth: overlap@k, sign agreement, rank correlation, and bootstrap CIs. **Meaningful only if the model was trained treatment-aware (`use_treatment=True`) and long enough** — the demo checkpoint will be weak.

In [ ]:
from microbiomepfn.prior_treatment import sample_dataset_with_treatment, TreatmentPriorConfig
from microbiomepfn.interpret import (
    counterfactual_treatment_effect, bootstrap_treatment_ranking, rank_table,
)
from scipy.stats import spearmanr

tcfg = TreatmentPriorConfig()
tcfg.n_samples_range = (120, 140)
tcfg.n_taxa_range = (150, 200)
tcfg.p_treatment_study = 1.0  # force a treatment
ds = sample_dataset_with_treatment(cfg=tcfg, rng=np.random.default_rng(11))
treat_idx = ds.hyperparams["treatment_cov_idx"]
true_eff = ds.true_effects[treat_idx]            # (T,)
responsive = true_eff != 0
print(f"treatment at col {treat_idx} | {int(responsive.sum())}/{len(true_eff)} responsive taxa "
      f"| n_levels={ds.hyperparams['treatment_n_levels']} asym={ds.hyperparams['treatment_asymmetry']:.2f}")

cte = counterfactual_treatment_effect(model, ds, treatment_col=treat_idx, cfg=cfg,
                                      reference_level=0, device=torch.device(device),
                                      rng=np.random.default_rng(0))
pred_mag = cte["effect_magnitudes"]              # (T,)
pred_lfc0 = cte["per_level_log_fold_change"][0]  # (T,)

k = int(responsive.sum())
true_top = set(np.argsort(-np.abs(true_eff))[:k].tolist())
pred_top = set(cte["ranked_taxa"][:k].tolist())
overlap = len(true_top & pred_top) / max(k, 1)
chance = k / len(true_eff)
sign_agree = (np.sign(pred_lfc0[responsive]) == np.sign(true_eff[responsive])).mean()
rho_mag, _ = spearmanr(pred_mag, np.abs(true_eff))
print(f"overlap@{k}                       : {overlap:.2f}  (chance ~ {chance:.2f})")
print(f"sign agreement (responsive taxa) : {sign_agree:.2f}")
print(f"Spearman(|pred|, |true|)          : {rho_mag:.3f}")

In [ ]:
boot = bootstrap_treatment_ranking(model, ds, treatment_col=treat_idx, cfg=cfg,
                                   n_boot=30, device=torch.device(device), seed=0, verbose=False)
n_sig = int(((boot["lo"] > 0) | (boot["hi"] < 0)).sum())
print(f"taxa with 90% bootstrap CI excluding zero: {n_sig}\n")
print(rank_table(boot["median"], top_k=15, ci=(boot["lo"], boot["hi"]),
                 title="Top responsive taxa (bootstrap median + 90% CI)"))

plt.figure(figsize=(5, 5))
plt.scatter(true_eff[~responsive], pred_lfc0[~responsive], s=10, alpha=0.4, label="non-responsive")
plt.scatter(true_eff[responsive], pred_lfc0[responsive], s=14, label="responsive")
plt.axhline(0, color="k", lw=0.5); plt.axvline(0, color="k", lw=0.5)
plt.xlabel("true effect (prior)"); plt.ylabel("predicted log-fold-change")
plt.legend(); plt.title("Treatment effect: predicted vs ground truth"); plt.show()

## Interpreting these results

- The standalone **demo checkpoint** is trained for only a few thousand steps. Expect it to modestly beat the baseline, pass the structural checks (4 & 5) *exactly*, and show weak-to-moderate treatment recovery. Point `CHECKPOINT` at a properly trained model for real numbers.
- **Experiments 4 & 5 are architectural invariants** — they must pass regardless of training quality. A failure there is a genuine regression, not a training-budget issue.
- **Treatment recovery (6)** depends on training the model treatment-aware (`use_treatment=True`). Per the README, confirm hits against conventional DA (MaAsLin2 / ANCOM-BC2) before trusting them.